# ModelNet40 — Exploratory Data Analysis
### WeightedNode Team | Uncertainty-Guided Spreading Activation

**Goal of this notebook:**  
Understand the ModelNet40 dataset deeply before building any graph or GNN.  
We produce 7 key artifacts that directly inform our graph construction and spreading activation design.

---
**Artifacts we build:**
1. Class distribution (train vs test imbalance)
2. Point cloud 3D visualizations per class
3. Per-class XYZ bounding box statistics
4. Normal vector distribution per class
5. kNN graph statistics at k=20 (degree, edge length, diameter)
6. Point density heatmaps (2D projections)
7. Inter-class similarity matrix (Chamfer distance)

---

## 0. Environment Setup & Imports

In [ ]:
# Run this once to install dependencies
# If you are on a fresh environment / Colab
!pip install torch torchvision --quiet
!pip install torch-geometric --quiet
!pip install torch-scatter torch-sparse torch-cluster --quiet
!pip install open3d --quiet          # for 3D visualization
!pip install matplotlib seaborn scipy tqdm --quiet

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import seaborn as sns
from mpl_toolkits.mplot3d import Axes3D
from collections import Counter, defaultdict
from tqdm import tqdm
import scipy.spatial as spatial

import torch
import torch_geometric
from torch_geometric.datasets import ModelNet
from torch_geometric.transforms import SamplePoints, NormalizeScale, KNNGraph
import torch_geometric.transforms as T
from torch_geometric.data import DataLoader
from torch_cluster import knn_graph

print(f"PyTorch version      : {torch.__version__}")
print(f"PyG version          : {torch_geometric.__version__}")
print(f"CUDA available       : {torch.cuda.is_available()}")

---
## 1. Load ModelNet40

PyG will download and preprocess ModelNet40 automatically.
We use `ModelNet('40', ...)` for the full 40-class version.

**Key transform choices:**
- `SamplePoints(1024, include_normals=True)` — samples exactly 1024 points from the mesh surface, including surface normals → gives us our 6D node features `[x,y,z,nx,ny,nz]`
- `NormalizeScale()` — centers the object and scales to unit sphere (standard preprocessing, used by DGCNN, PointNet++)

**Literature justification:**  
Wang et al. (DGCNN, TOG 2019) use 1024 points with normals for the ModelNet40 classification benchmark.

In [ ]:
DATA_ROOT = './data/ModelNet40'   # change this if needed
NUM_POINTS = 1024

# Transform: sample 1024 points with normals, then normalize to unit sphere
pre_transform = T.Compose([
    SamplePoints(NUM_POINTS, include_normals=True),
    NormalizeScale()
])

train_dataset = ModelNet(
    root=DATA_ROOT,
    name='40',
    train=True,
    pre_transform=pre_transform
)

test_dataset = ModelNet(
    root=DATA_ROOT,
    name='40',
    train=False,
    pre_transform=pre_transform
)

print(f"Train samples        : {len(train_dataset)}")
print(f"Test samples         : {len(test_dataset)}")
print(f"Number of classes    : {train_dataset.num_classes}")
print(f"Class names          : {train_dataset.categories}")
print()

# Inspect a single sample
sample = train_dataset[0]
print(f"Sample data object   : {sample}")
print(f"pos shape (xyz)      : {sample.pos.shape}   # [1024, 3]")
print(f"norm shape (normals) : {sample.norm.shape}  # [1024, 3]")
print(f"label                : {sample.y.item()} = '{train_dataset.categories[sample.y.item()]}'")

---
## Artifact 1 — Class Distribution: Train vs Test Imbalance

**Why this matters for our project:**  
ModelNet40 is known to be imbalanced. Minority classes will have higher *epistemic* uncertainty
(model ignorance from lack of data) vs majority classes which may have higher *aleatoric* uncertainty
(intrinsic geometric complexity). Our spreading activation must not confuse these two.

**TODO for you:** After running this cell, identify:
- The top-5 majority classes
- The top-5 minority classes
- The imbalance ratio (max_count / min_count)
- Whether the train/test split preserves class proportions

In [ ]:
# Count samples per class
train_counts = Counter()
test_counts  = Counter()

for data in tqdm(train_dataset, desc='Counting train'):
    train_counts[train_dataset.categories[data.y.item()]] += 1

for data in tqdm(test_dataset, desc='Counting test'):
    test_counts[test_dataset.categories[data.y.item()]] += 1

# Sort by train count descending
sorted_classes = sorted(train_counts.keys(), key=lambda c: train_counts[c], reverse=True)
train_vals = [train_counts[c] for c in sorted_classes]
test_vals  = [test_counts[c]  for c in sorted_classes]

# Imbalance ratio
imbalance_ratio = max(train_vals) / min(train_vals)
print(f"Imbalance ratio (max/min train samples): {imbalance_ratio:.1f}x")
print(f"Max class: {sorted_classes[0]} ({train_vals[0]} samples)")
print(f"Min class: {sorted_classes[-1]} ({train_vals[-1]} samples)")

# Plot
fig, axes = plt.subplots(1, 2, figsize=(22, 7))

x = np.arange(len(sorted_classes))
width = 0.4
axes[0].bar(x - width/2, train_vals, width, label='Train', color='steelblue', alpha=0.85)
axes[0].bar(x + width/2, test_vals,  width, label='Test',  color='tomato',    alpha=0.85)
axes[0].set_xticks(x)
axes[0].set_xticklabels(sorted_classes, rotation=90, fontsize=8)
axes[0].set_ylabel('Sample Count')
axes[0].set_title('ModelNet40 — Class Distribution (Train vs Test)')
axes[0].legend()
axes[0].axhline(np.mean(train_vals), color='navy', linestyle='--', alpha=0.5, label='Mean train')

# Train/test ratio per class
ratios = [train_counts[c] / (test_counts[c] + 1e-6) for c in sorted_classes]
axes[1].bar(x, ratios, color='mediumseagreen', alpha=0.8)
axes[1].set_xticks(x)
axes[1].set_xticklabels(sorted_classes, rotation=90, fontsize=8)
axes[1].axhline(4.0, color='red', linestyle='--', alpha=0.7, label='Expected 4:1 ratio')
axes[1].set_ylabel('Train / Test Ratio')
axes[1].set_title('Train-to-Test Ratio Per Class (Should be ~4)')
axes[1].legend()

plt.tight_layout()
plt.savefig('artifact1_class_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: artifact1_class_distribution.png")

---
## Artifact 2 — 3D Point Cloud Visualization Per Class

**Why this matters:**  
Visually inspecting the point clouds tells you:
- How much intra-class geometric variance exists (plant vs cone)
- Which classes have flat surfaces (table, desk) → clustered normals
- Which classes have complex topology (chair, lamp) → diverse point distributions

**TODO:** After running, identify 3 classes with HIGH intra-class variance and 3 with LOW variance.
These will be hard vs easy cases for your uncertainty estimator.

In [ ]:
# Helper: get first N samples per class
def get_samples_per_class(dataset, n_per_class=3):
    class_samples = defaultdict(list)
    for data in dataset:
        cls = dataset.categories[data.y.item()]
        if len(class_samples[cls]) < n_per_class:
            class_samples[cls].append(data)
        if all(len(v) >= n_per_class for v in class_samples.values()):
            break
    return class_samples

class_samples = get_samples_per_class(train_dataset, n_per_class=2)

# Visualize a subset of classes (change this list to explore others)
CLASSES_TO_PLOT = ['chair', 'airplane', 'lamp', 'table', 'plant', 'cone', 
                   'guitar', 'car', 'laptop', 'flower_pot']

fig = plt.figure(figsize=(20, 8))
for idx, cls_name in enumerate(CLASSES_TO_PLOT):
    if cls_name not in class_samples:
        continue
    data = class_samples[cls_name][0]
    pts  = data.pos.numpy()    # [1024, 3]

    ax = fig.add_subplot(2, 5, idx + 1, projection='3d')
    ax.scatter(pts[:, 0], pts[:, 1], pts[:, 2],
               c=pts[:, 2],          # color by height (Z axis)
               cmap='viridis', s=1.5, alpha=0.7)
    ax.set_title(cls_name, fontsize=10, fontweight='bold')
    ax.set_xticks([]); ax.set_yticks([]); ax.set_zticks([])
    ax.set_xlabel('X', fontsize=7); ax.set_ylabel('Y', fontsize=7)

plt.suptitle('ModelNet40 — Point Cloud Visualizations (colored by Z/height)', 
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('artifact2_pointcloud_viz.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: artifact2_pointcloud_viz.png")

In [ ]:
# Intra-class variance: compare 2 samples of the same class side-by-side
# Try 'chair', 'plant', 'cone' to see low vs high intra-class variance
COMPARE_CLASS = 'chair'   # <-- change this to explore

samples_to_compare = class_samples[COMPARE_CLASS]
fig = plt.figure(figsize=(12, 5))
for i, data in enumerate(samples_to_compare):
    pts = data.pos.numpy()
    ax  = fig.add_subplot(1, 2, i+1, projection='3d')
    ax.scatter(pts[:,0], pts[:,1], pts[:,2], c=pts[:,2], cmap='plasma', s=2)
    ax.set_title(f'{COMPARE_CLASS} — Sample {i+1}', fontweight='bold')
    ax.set_xticks([]); ax.set_yticks([]); ax.set_zticks([])

plt.suptitle(f'Intra-class variance for: {COMPARE_CLASS}', fontsize=12)
plt.tight_layout()
plt.savefig(f'artifact2b_intraclass_{COMPARE_CLASS}.png', dpi=150, bbox_inches='tight')
plt.show()

---
## Artifact 3 — Per-class XYZ Bounding Box Statistics

**Why this matters:**  
- **Spread in Z** → height variance → tall objects (lamp, plant) vs flat objects (table)
- **Spread in XY** → lateral extent → wide objects (sofa, bookshelf) vs thin (guitar)
- High std in any axis = more geometric variation within the class  
- This predicts how large the kNN graph neighborhoods will be and how far spreading activation will propagate

**Literature mapping:**  
PointNet++ (Qi et al., NeurIPS 2017) normalizes to unit sphere precisely because of this variance — after `NormalizeScale()`, all objects fit in [-1,1]³. We measure AFTER normalization.

In [ ]:
# Compute bbox stats per class (mean and std of point spread per axis)
# Takes a few minutes — runs over full training set

bbox_stats = defaultdict(lambda: {'x_range': [], 'y_range': [], 'z_range': [],
                                   'x_std':   [], 'y_std':   [], 'z_std':   []})

for data in tqdm(train_dataset, desc='Computing bbox stats'):
    cls = train_dataset.categories[data.y.item()]
    pts = data.pos.numpy()
    bbox_stats[cls]['x_range'].append(pts[:,0].max() - pts[:,0].min())
    bbox_stats[cls]['y_range'].append(pts[:,1].max() - pts[:,1].min())
    bbox_stats[cls]['z_range'].append(pts[:,2].max() - pts[:,2].min())
    bbox_stats[cls]['x_std'].append(pts[:,0].std())
    bbox_stats[cls]['y_std'].append(pts[:,1].std())
    bbox_stats[cls]['z_std'].append(pts[:,2].std())

# Summarize
categories = sorted(bbox_stats.keys())
mean_x_range = [np.mean(bbox_stats[c]['x_range']) for c in categories]
mean_y_range = [np.mean(bbox_stats[c]['y_range']) for c in categories]
mean_z_range = [np.mean(bbox_stats[c]['z_range']) for c in categories]
mean_z_std   = [np.mean(bbox_stats[c]['z_std'])   for c in categories]

x_idx = np.arange(len(categories))

fig, axes = plt.subplots(2, 1, figsize=(22, 10))

# Bounding box ranges
width = 0.28
axes[0].bar(x_idx - width,   mean_x_range, width, label='X range', color='steelblue', alpha=0.8)
axes[0].bar(x_idx,           mean_y_range, width, label='Y range', color='tomato',    alpha=0.8)
axes[0].bar(x_idx + width,   mean_z_range, width, label='Z range', color='goldenrod', alpha=0.8)
axes[0].set_xticks(x_idx)
axes[0].set_xticklabels(categories, rotation=90, fontsize=8)
axes[0].set_ylabel('Mean Bounding Box Range (normalized)')
axes[0].set_title('Per-class Bounding Box Ranges (XYZ) — after NormalizeScale')
axes[0].legend()

# Z-std (height variation)
colors = cm.coolwarm(np.array(mean_z_std) / max(mean_z_std))
axes[1].bar(x_idx, mean_z_std, color=colors, alpha=0.85)
axes[1].set_xticks(x_idx)
axes[1].set_xticklabels(categories, rotation=90, fontsize=8)
axes[1].set_ylabel('Mean Z-axis Std Dev')
axes[1].set_title('Per-class Z-axis Point Spread (height diversity) — red = high variance')

plt.tight_layout()
plt.savefig('artifact3_bbox_stats.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: artifact3_bbox_stats.png")

---
## Artifact 4 — Normal Vector Distribution Per Class

**Why this matters for our 6D features:**  
The surface normals `[nx, ny, nz]` are the 4th–6th dimensions of our node features.  
- **Flat surfaces** (table, desk, floor): normals cluster tightly around a single direction (e.g., [0,0,1])
- **Complex surfaces** (chair, plant, lamp): normals spread across the unit sphere

**Literature mapping:**  
Att-AdaptNet (2024) uses `dot(n_i, n_j)` as an edge feature to measure surface smoothness.  
A high dot product along an edge = smooth surface continuation = weaker activation barrier.  
A low dot product = surface discontinuity = strong activation signal (boundary/corner).

In [ ]:
# Sample a few classes and plot their normal vector distributions on the unit sphere
NORMAL_CLASSES = ['table', 'chair', 'plant', 'airplane', 'cone', 'lamp']

# Collect normals for each class (first 5 samples each)
class_normals = defaultdict(list)
counts_needed = {c: 5 for c in NORMAL_CLASSES}

for data in train_dataset:
    cls = train_dataset.categories[data.y.item()]
    if cls in counts_needed and counts_needed[cls] > 0:
        class_normals[cls].append(data.norm.numpy())  # [1024, 3]
        counts_needed[cls] -= 1
    if all(v == 0 for v in counts_needed.values()):
        break

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes = axes.flatten()

for idx, cls_name in enumerate(NORMAL_CLASSES):
    normals = np.concatenate(class_normals[cls_name], axis=0)  # [N*1024, 3]
    ax = axes[idx]

    # Plot normal directions as a 2D histogram of (nx, nz) — front view
    h = ax.hist2d(normals[:,0], normals[:,2],
                  bins=60, cmap='hot',
                  range=[[-1,1],[-1,1]])
    fig.colorbar(h[3], ax=ax, shrink=0.8)
    ax.set_title(f'{cls_name}', fontsize=11, fontweight='bold')
    ax.set_xlabel('nx')
    ax.set_ylabel('nz (vertical)')

    # Compute normal entropy as a diversity score
    # High entropy = normals spread across sphere = complex surface
    hist, _ = np.histogramdd(normals[:, :2], bins=20, range=[[-1,1]]*2)
    hist = hist / hist.sum() + 1e-10
    entropy = -np.sum(hist * np.log(hist))
    ax.set_xlabel(f'nx  |  Normal entropy: {entropy:.2f}', fontsize=9)

plt.suptitle('Normal Vector Distribution (nx vs nz) — hot = high density\n'
             'Flat objects cluster at center; complex objects spread outward',
             fontsize=12)
plt.tight_layout()
plt.savefig('artifact4_normal_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: artifact4_normal_distribution.png")
print()
print("TODO: Classes with HIGH normal entropy = geometrically complex = harder for classifier")
print("TODO: These are the classes where spreading activation will find the most interesting seeds")

---
## Artifact 5 — kNN Graph Statistics at k=20

**Why this matters:**  
Before our GNN sees any data, we need to understand the graph topology our construction produces.

**Key metrics:**
- **Mean edge length** per class → how local/global the neighborhoods are
- **Degree distribution** → should be constant at k=20 for all nodes (sanity check)
- **Graph diameter** → max shortest path → how many hops for spreading activation to cover the full object

**Literature mapping:**  
DGCNN (Wang et al., 2019) uses k=20 as the standard for ModelNet40.  
MLGCN (2024) shows that k=20 graphs can be precomputed once and shared across layers — we can exploit this.

In [ ]:
from torch_cluster import knn_graph

K = 20  # DGCNN standard

# Compute graph stats for a subset of classes (full dataset takes long)
GRAPH_STATS_CLASSES = ['chair', 'airplane', 'plant', 'table', 'cone', 'lamp', 'guitar', 'car']
graph_stats = defaultdict(lambda: {'edge_lengths': [], 'degrees': [], 'n_nodes': []})

counts_done = defaultdict(int)
MAX_PER_CLASS = 30  # use 30 samples per class for speed

for data in tqdm(train_dataset, desc='Building kNN graphs'):
    cls = train_dataset.categories[data.y.item()]
    if cls not in GRAPH_STATS_CLASSES:
        continue
    if counts_done[cls] >= MAX_PER_CLASS:
        continue

    pts = data.pos  # [1024, 3]

    # Build kNN graph using torch_cluster
    edge_index = knn_graph(pts, k=K, loop=False)  # [2, N*K]

    # Edge lengths
    src, dst = edge_index
    edge_vecs = pts[dst] - pts[src]               # [N*K, 3]
    edge_lens = edge_vecs.norm(dim=1).numpy()      # [N*K]
    graph_stats[cls]['edge_lengths'].extend(edge_lens.tolist())

    # Node degrees (should all be K for kNN)
    degree = torch.zeros(pts.shape[0], dtype=torch.long)
    degree.scatter_add_(0, src, torch.ones(src.shape[0], dtype=torch.long))
    graph_stats[cls]['degrees'].extend(degree.numpy().tolist())
    graph_stats[cls]['n_nodes'].append(pts.shape[0])

    counts_done[cls] += 1

# Plot edge length distributions per class
fig, axes = plt.subplots(2, 4, figsize=(20, 8))
axes = axes.flatten()

for idx, cls_name in enumerate(GRAPH_STATS_CLASSES):
    el = graph_stats[cls_name]['edge_lengths']
    ax = axes[idx]
    ax.hist(el, bins=60, color='steelblue', alpha=0.8, edgecolor='white', linewidth=0.3)
    ax.set_title(f'{cls_name}\nmean={np.mean(el):.3f} std={np.std(el):.3f}', fontsize=9)
    ax.set_xlabel('Edge length (normalized)', fontsize=8)
    ax.set_ylabel('Count', fontsize=8)
    ax.axvline(np.mean(el), color='red', linestyle='--', alpha=0.8)

plt.suptitle(f'kNN Graph Edge Length Distribution per Class (k={K})\n'
             'Short edges = dense/compact geometry; Long edges = sparse/spread-out geometry',
             fontsize=11)
plt.tight_layout()
plt.savefig('artifact5_knn_edge_lengths.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: artifact5_knn_edge_lengths.png")

In [ ]:
# Summary table: mean edge length per class
print(f"{'Class':<15} {'Mean Edge Len':>15} {'Std Edge Len':>13} {'Interpretation':<30}")
print('-' * 75)
for cls_name in sorted(GRAPH_STATS_CLASSES,
                        key=lambda c: np.mean(graph_stats[c]['edge_lengths'])):
    el = graph_stats[cls_name]['edge_lengths']
    mean_el = np.mean(el)
    std_el  = np.std(el)
    interp  = 'COMPACT geometry' if mean_el < 0.1 else ('SPREAD geometry' if mean_el > 0.2 else 'MEDIUM geometry')
    print(f"{cls_name:<15} {mean_el:>15.4f} {std_el:>13.4f}   {interp}")

---
## Artifact 6 — Point Density Heatmaps (2D Projections)

**Why this matters:**  
After normalization, where do points actually cluster in 3D space?  
This tells you where spreading activation is most likely to **seed** (high density = high information).

We project onto 3 planes: XY (top view), XZ (front view), YZ (side view).

**TODO:** After running, identify:
- Which classes have symmetric density maps (cone, sphere) vs asymmetric (chair leg vs seat)
- These asymmetric regions = candidate seeds for spreading activation
- Check if pre-alignment is visible (all chairs point same direction?)

In [ ]:
DENSITY_CLASSES = ['chair', 'airplane', 'table', 'lamp']
BINS = 50

# Aggregate points across multiple samples per class
class_all_pts = defaultdict(list)
counts_for_density = defaultdict(int)
MAX_DENSITY_SAMPLES = 50

for data in tqdm(train_dataset, desc='Aggregating points for density'):
    cls = train_dataset.categories[data.y.item()]
    if cls not in DENSITY_CLASSES:
        continue
    if counts_for_density[cls] >= MAX_DENSITY_SAMPLES:
        continue
    class_all_pts[cls].append(data.pos.numpy())
    counts_for_density[cls] += 1

fig, axes = plt.subplots(len(DENSITY_CLASSES), 3, figsize=(15, 4 * len(DENSITY_CLASSES)))
PROJECTIONS = [('XY (top)',   0, 1), ('XZ (front)', 0, 2), ('YZ (side)', 1, 2)]

for row, cls_name in enumerate(DENSITY_CLASSES):
    all_pts = np.concatenate(class_all_pts[cls_name], axis=0)  # [N*1024, 3]

    for col, (proj_name, xi, yi) in enumerate(PROJECTIONS):
        ax = axes[row][col]
        h = ax.hist2d(all_pts[:, xi], all_pts[:, yi],
                      bins=BINS, cmap='inferno',
                      range=[[-1,1],[-1,1]])
        fig.colorbar(h[3], ax=ax, shrink=0.8)
        ax.set_title(f'{cls_name} — {proj_name}', fontsize=9, fontweight='bold')
        ax.set_xlabel(['X','X','Y'][col], fontsize=8)
        ax.set_ylabel(['Y','Z','Z'][col], fontsize=8)
        ax.set_aspect('equal')

plt.suptitle('Point Density Heatmaps (aggregated across 50 samples per class)\n'
             'Bright = high density = candidate spreading activation seed regions',
             fontsize=12)
plt.tight_layout()
plt.savefig('artifact6_density_heatmaps.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: artifact6_density_heatmaps.png")

---
## Artifact 7 — Inter-class Similarity Matrix (Chamfer Distance)

**Why this matters:**  
The Chamfer distance between two point clouds measures how geometrically similar they are.  
Low Chamfer distance between two classes = they look geometrically similar = your classifier will confuse them.

**These confusable pairs are exactly where your uncertainty estimator will produce high entropy.**  
This is your ground truth for evaluating whether spreading activation seeds correctly.

**Literature mapping:**  
Common confusable pairs in ModelNet40 (widely reported in literature):
- dresser ↔ nightstand
- flower_pot ↔ vase  
- desk ↔ table
- sofa ↔ bed

**Note:** Full Chamfer matrix over all 40 classes is expensive. We compute a subset here.

In [ ]:
from scipy.spatial import cKDTree

def chamfer_distance(pc1: np.ndarray, pc2: np.ndarray) -> float:
    """
    Chamfer distance between two point clouds.
    pc1, pc2: [N, 3] and [M, 3] numpy arrays
    Returns: scalar distance (lower = more similar)
    """
    tree1 = cKDTree(pc1)
    tree2 = cKDTree(pc2)
    d1, _ = tree2.query(pc1, k=1)  # for each pt in pc1, nearest in pc2
    d2, _ = tree1.query(pc2, k=1)  # for each pt in pc2, nearest in pc1
    return float(np.mean(d1**2) + np.mean(d2**2))

# Select a subset of interesting/potentially confusable classes
CHAMFER_CLASSES = [
    'chair', 'sofa', 'bed', 'table', 'desk',
    'dresser', 'night_stand', 'bookshelf', 'wardrobe',
    'flower_pot', 'vase', 'plant', 'cup', 'bowl'
]
# Filter to only those that exist in the dataset
CHAMFER_CLASSES = [c for c in CHAMFER_CLASSES if c in train_dataset.categories]

# Get ONE representative point cloud per class (mean of 10 samples)
class_rep_pts = {}
counts_for_chamfer = defaultdict(int)
accum_pts = defaultdict(list)

for data in train_dataset:
    cls = train_dataset.categories[data.y.item()]
    if cls not in CHAMFER_CLASSES:
        continue
    if counts_for_chamfer[cls] >= 10:
        continue
    accum_pts[cls].append(data.pos.numpy())
    counts_for_chamfer[cls] += 1

for cls in CHAMFER_CLASSES:
    if accum_pts[cls]:
        # Use first sample as representative (or mean if same #points)
        class_rep_pts[cls] = accum_pts[cls][0]

# Compute Chamfer matrix
valid_classes = [c for c in CHAMFER_CLASSES if c in class_rep_pts]
N = len(valid_classes)
chamfer_matrix = np.zeros((N, N))

print(f"Computing {N}x{N} Chamfer distance matrix...")
for i, ci in enumerate(tqdm(valid_classes)):
    for j, cj in enumerate(valid_classes):
        if i == j:
            chamfer_matrix[i, j] = 0.0
        elif j > i:
            d = chamfer_distance(class_rep_pts[ci], class_rep_pts[cj])
            chamfer_matrix[i, j] = d
            chamfer_matrix[j, i] = d

# Plot
fig, ax = plt.subplots(figsize=(12, 10))
mask = np.eye(N, dtype=bool)  # mask diagonal
sns.heatmap(
    chamfer_matrix,
    xticklabels=valid_classes,
    yticklabels=valid_classes,
    cmap='YlOrRd_r',          # dark = low distance = similar
    annot=True,
    fmt='.3f',
    linewidths=0.5,
    ax=ax,
    mask=mask
)
ax.set_title('Inter-class Chamfer Distance Matrix\n'
             'Dark = geometrically SIMILAR (high confusion risk)\n'
             'These pairs will produce high uncertainty in your spreading activation seeder',
             fontsize=11)
plt.xticks(rotation=45, ha='right', fontsize=9)
plt.yticks(rotation=0, fontsize=9)
plt.tight_layout()
plt.savefig('artifact7_chamfer_matrix.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: artifact7_chamfer_matrix.png")

# Print top-5 most similar pairs
print("\nTop-5 Most Geometrically Similar (Confusable) Class Pairs:")
pairs = []
for i in range(N):
    for j in range(i+1, N):
        pairs.append((chamfer_matrix[i,j], valid_classes[i], valid_classes[j]))
pairs.sort()
for d, ci, cj in pairs[:5]:
    print(f"  {ci:<15} ↔  {cj:<15}  Chamfer dist = {d:.4f}")

---
## Summary — What We Learned & Design Decisions

Fill this in after running all cells above. Use this as your EDA report section.

### 1. Class Imbalance
- Imbalance ratio: **[fill from Artifact 1]**
- Majority classes (top 5): **[fill]**  
- Minority classes (bottom 5): **[fill]**  
- Impact on spreading activation: high epistemic uncertainty likely for minority classes regardless of geometry

### 2. Geometric Complexity by Class
- High intra-class variance (hard for GNN): **[fill from Artifact 2 + 3]**  
- Low intra-class variance (easy): **[fill]**  

### 3. Normal Vector Analysis → 6D Feature Justification
- Classes with high normal entropy (complex surfaces): **[fill from Artifact 4]**  
- These classes benefit most from `[nx,ny,nz]` in the node feature vector  
- Edge feature `dot(n_i, n_j)` will be most informative at **[these classes]**

### 4. kNN Graph Properties at k=20
- Mean edge length range across classes: **[fill from Artifact 5]**  
- Classes where spreading activation will propagate in tight clusters: **[fill]**  
- Classes needing more hops to cover: **[fill]**

### 5. Density Structure → Seed Region Candidates
- Classes with clear high-density regions (good seeds): **[fill from Artifact 6]**  
- Classes with uniform density (spreading activation will spread evenly): **[fill]**

### 6. Confusion-Prone Class Pairs → Uncertainty Calibration Targets
- Top confusable pairs from Chamfer analysis: **[fill from Artifact 7]**  
- These are your hardest test cases for the uncertainty estimator

---
### Next Steps
1. Build the kNN graph construction module (k=20, 6D node features, 4D edge features)
2. Implement a simple GCN/GIN baseline on ModelNet40 to establish accuracy numbers
3. Add uncertainty estimator (MC Dropout) per node
4. Implement spreading activation on the graph
5. Prune to sparse graph → pass to ViG block
6. Compare vs DGCNN baseline at same k=20